In [98]:
import pandas as pd
import os
import re
from datetime import datetime, timedelta

In [99]:
INPUT_FILES_DIR = r"dataset\output\temporal_meta"
TOTAL_CLUSTERS = len(os.listdir(INPUT_FILES_DIR))

print(f"Total Clusters: {TOTAL_CLUSTERS}")

Total Clusters: 5


In [100]:
OUTPUT_PATH = r"dataset\output\temporal_sentence"

In [101]:
for files in os.listdir(OUTPUT_PATH):
    os.remove(os.path.join(OUTPUT_PATH,files))

In [102]:
SENT_COL = "sentence"
ARTICLE_COL = "article_id"
PUBDATE_COL = "pub_date"
SENT_ORDER_COL = "sent_id"

In [103]:
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")
MONTH_NAME_RE = re.compile(r"\b(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)\b", re.IGNORECASE)
DAY_MONTH_YEAR_RE = re.compile(r"\b(\d{1,2})[\/\-\s](\d{1,2})[\/\-\s]((?:19|20)\d{2})\b")
MONTH_YEAR_RE = re.compile(r"\b(" + "|".join([m for m in ["January","February","March","April","May","June","July","August","September","October","November","December"]]) + r")\s+((?:19|20)\d{2})\b", re.IGNORECASE)

RELATIVE_KEYWORDS = {
    r"\b(yesterday)\b": lambda ref: ref - timedelta(days=1),
    r"\b(today|currently|at present|now)\b": lambda ref: ref,
    r"\b(tomorrow)\b": lambda ref: ref + timedelta(days=1),
    r"\b(last year|previous year|the year before)\b": lambda ref: ref.replace(year=ref.year-1),
    r"\b(next year|following year)\b": lambda ref: ref.replace(year=ref.year+1),
    r"\b(last month)\b": lambda ref: (ref - pd.DateOffset(months=1)).to_pydatetime(),
    r"\b(next month)\b": lambda ref: (ref + pd.DateOffset(months=1)).to_pydatetime(),
    r"\b(\d+)\s+years?\s+ago\b": None,
    r"\b(\d+)\s+months?\s+ago\b": None,
    r"\b(earlier this year|earlier this month)\b": lambda ref: ref,
    r"\b(recently|recent)\b": lambda ref: ref,
    r"\b(announced|said|reported|stated|revealed|claimed)\b": None
}

SEQUENCE_MARKERS = {
    r"\b(first|initially|to begin with|at first)\b": 1,
    r"\b(then|next|afterwards|after that|subsequently)\b": 2,
    r"\b(later|following this|following that)\b": 3,
    r"\b(finally|lastly|in conclusion)\b": 4,
    r"\b(previously|earlier)\b": 0.5,
    r"\b(currently|now)\b": 2.5
}

In [104]:

def extract_year(text):
    m = YEAR_RE.search(text)
    if m:
        return int(m.group(0))
    return None

def extract_day_month_year(text):
    m = DAY_MONTH_YEAR_RE.search(text)
    if m:
        d, mo, y = m.groups()
        try:
            return datetime(int(y), int(mo), int(d))
        except:
            return None
    return None

def extract_month_year(text):
    m = MONTH_YEAR_RE.search(text)
    if m:
        month_name, year = m.groups()
        try:
            dt = datetime.strptime(f"{month_name} {year}", "%B %Y")
            return dt
        except:
            try:
                dt = datetime.strptime(f"{month_name} {year}", "%b %Y")
                return dt
            except:
                return None
    return None

def extract_relative(text, ref_date):
    text_low = text.lower()
    for pat, func in RELATIVE_KEYWORDS.items():
        if re.search(pat, text_low):
            if func is None:
                m = re.search(r"(\d+)\s+years?\s+ago", text_low)
                if m:
                    years = int(m.group(1))
                    try:
                        return ref_date.replace(year=ref_date.year - years)
                    except:
                        return ref_date - timedelta(days=years*365)
                m2 = re.search(r"(\d+)\s+months?\s+ago", text_low)
                if m2:
                    months = int(m2.group(1))
                    return (ref_date - pd.DateOffset(months=months)).to_pydatetime()
                return None
            else:
                try:
                    return func(ref_date)
                except Exception:
                    return None
    return None

def seq_marker_score(text):
    text_low = text.lower()
    scores = []
    for pat, score in SEQUENCE_MARKERS.items():
        if re.search(pat, text_low):
            scores.append(score)
    if scores:
        return min(scores)
    return None

def analyze_row(row):
    text = str(row[SENT_COL])
    pub = row["pub_date"]
    if pd.isna(pub):
        pub = datetime.now()
    dm = extract_day_month_year(text)
    if dm is not None:
        return {"date": dm, "source": "explicit_dm"}
    my = extract_month_year(text)
    if my is not None:
        return {"date": my, "source": "month_year"}
    y = extract_year(text)
    if y is not None:
        try:
            dt = datetime(y, 6, 15)
            return {"date": dt, "source": "year_only"}
        except:
            pass

    rel = extract_relative(text, pub)
    if rel is not None:
        return {"date": rel, "source": "relative"}

    seq = seq_marker_score(text)
    if seq is not None:
        pseudo = pub + timedelta(days=0)
        offset_days = int((seq - 2) * 2)
        pseudo = pseudo + timedelta(days=offset_days)
        return {"date": pseudo, "source": "sequence_marker", "seq_score": seq}

    return {"date": None, "source": None}

def compute_temporal_score(row):
    if pd.notna(row["_extracted_dt"]):
        return row["_extracted_dt"].timestamp()
    base = row["pub_date"]
    if pd.isna(base):
        base = datetime.now()
    offset = int(row[SENT_ORDER_COL]) * 60
    return base.timestamp() + offset


In [105]:
for c in range(TOTAL_CLUSTERS):
    file_name = f"cluster{c}_temporalSort_metaData.csv"
    INPUT_PATH = os.path.join(INPUT_FILES_DIR,file_name)
    
    df = pd.read_csv(INPUT_PATH)
    df = df.reset_index(drop=True)
    df['sent_id'] = df.index
    df['sentence'] = df['sentence'].astype(str).str.strip()
    
    print(f"Cluster {c}, total sentences: {len(df)}")
    
    df["pub_date"] = pd.to_datetime(df["pub_date"], errors="coerce")
    
    meta = df.apply(analyze_row, axis=1, result_type="expand")
    df["_extracted_dt"] = pd.to_datetime(meta["date"], errors="coerce",utc=True)
    df["_date_source"] = meta["source"]
    df["_seq_score"] = meta.get("seq_score")
    
    df["_temporal_score"] = df.apply(compute_temporal_score, axis=1)

    df_sorted = df.sort_values([ARTICLE_COL, "_temporal_score"])
    
    print("\nSummary of date source counts:")
    print(df_sorted["_date_source"].value_counts(dropna=False),end="\n-------------------------\n\n")
    
    df_sorted.to_csv(os.path.join(OUTPUT_PATH,f"cluster{c}_temporalSort.csv"), index=False)

Cluster 0, total sentences: 54

Summary of date source counts:
NaN                49
sequence_marker     2
relative            2
year_only           1
Name: _date_source, dtype: int64
-------------------------

Cluster 1, total sentences: 12

Summary of date source counts:
None          10
month_year     2
Name: _date_source, dtype: int64
-------------------------

Cluster 2, total sentences: 22



Summary of date source counts:
NaN                17
year_only           2
month_year          2
sequence_marker     1
Name: _date_source, dtype: int64
-------------------------

Cluster 3, total sentences: 11

Summary of date source counts:
NaN                9
sequence_marker    1
relative           1
Name: _date_source, dtype: int64
-------------------------

Cluster 4, total sentences: 7

Summary of date source counts:
None         5
year_only    1
relative     1
Name: _date_source, dtype: int64
-------------------------

